In [9]:
# One dict per run, every parameter written out. Change one parameter at a time.
EXPERIMENTS = [
    {"group": "baseline",
     "prompt": "calm lakeside evening ambient music, slow tempo around 60 BPM, soft piano and strings, atmospheric",
     "seed": 42, "duration": 10, "temperature": 1.0, "guidance": 3.0, "top_k": 250},
     {"group": "baseline",
          "prompt": "calm lakeside evening ambient music, slow tempo around 60 BPM, soft piano and strings, atmospheric",
          "seed": 40, "duration": 40, "temperature": 1.3, "guidance": 3.0, "top_k": 250}

    # Example: copy the baseline, remove the #, change one value (here: temperature).
    # {"group": "temperature",
    #  "prompt": "calm lakeside evening ambient music, slow tempo around 60 BPM, soft piano and strings, atmospheric",
    #  "seed": 42, "duration": 10, "temperature": 0.7, "guidance": 3.0, "top_k": 250},
]

In [10]:
import torch
from pathlib import Path
from IPython.display import Audio, display
from scipy.io import wavfile
from transformers import AutoProcessor, MusicgenForConditionalGeneration
from helpers import output_path, append_row

MODEL = "facebook/musicgen-small"
OUT_DIR = "outputs/audio"
CSV_PATH = "results/audio.csv"

processor = AutoProcessor.from_pretrained(MODEL, cache_dir="models")
model = MusicgenForConditionalGeneration.from_pretrained(MODEL, cache_dir="models").to("cuda")
RATE = model.config.audio_encoder.sampling_rate    # samples per second of the output
FRAME_RATE = model.config.audio_encoder.frame_rate  # tokens generated per second of audio
Path(OUT_DIR).mkdir(parents=True, exist_ok=True)
print("sampling rate:", RATE, "| frame rate:", FRAME_RATE)

Loading weights: 100%|██████████| 611/611 [00:02<00:00, 290.17it/s]


sampling rate: 32000 | frame rate: 50


In [11]:
new = skipped = 0
for e in EXPERIMENTS:
    path = output_path(e, OUT_DIR, "wav")
    if path.exists():
        skipped += 1
        print("SKIPPED", path.name)
    else:
        inputs = processor(text=[e["prompt"]], padding=True, return_tensors="pt").to("cuda")
        torch.manual_seed(e["seed"])  # generate() takes no generator, so each run reseeds torch
        audio = model.generate(**inputs, do_sample=True, max_new_tokens=int(e["duration"] * FRAME_RATE),
                               temperature=e["temperature"], guidance_scale=e["guidance"], top_k=e["top_k"])
        wavfile.write(path, RATE, audio[0, 0].cpu().numpy())
        append_row(CSV_PATH, e, path.name)
        new += 1
        print("NEW", path.name)
    display(Audio(str(path)))
print(f"{new} new, {skipped} skipped")

SKIPPED baseline_5acd3722.wav


NEW baseline_e7f98321.wav


1 new, 1 skipped
